In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/local/opt/openjdk@17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/01 21:13:02 WARN Utils: Your hostname, Jaimes-MacBook-Pro-3.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.10 instead (on interface en0)
26/03/01 21:13:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 21:13:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
ls -a data/pq/green/2020/01

./
../
._SUCCESS.crc
.part-00000-d72c6cf7-4b91-434c-89e1-0a6158908696-c000.snappy.parquet.crc
.part-00001-d72c6cf7-4b91-434c-89e1-0a6158908696-c000.snappy.parquet.crc
.part-00002-d72c6cf7-4b91-434c-89e1-0a6158908696-c000.snappy.parquet.crc
.part-00003-d72c6cf7-4b91-434c-89e1-0a6158908696-c000.snappy.parquet.crc
_SUCCESS
part-00000-d72c6cf7-4b91-434c-89e1-0a6158908696-c000.snappy.parquet
part-00001-d72c6cf7-4b91-434c-89e1-0a6158908696-c000.snappy.parquet
part-00002-d72c6cf7-4b91-434c-89e1-0a6158908696-c000.snappy.parquet
part-00003-d72c6cf7-4b91-434c-89e1-0a6158908696-c000.snappy.parquet


In [15]:
df_green = spark.read.option("recursiveFileLookup", "true") \
    .parquet("./data/pq/green/")

In [16]:
df_green.count()

2304517

In [17]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [18]:
df_yellow = spark.read.option("recursiveFileLookup", "true") \
    .parquet("./data/pq/yellow/")

In [19]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [20]:
common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

In [22]:
from pyspark.sql import functions as F

df_green_sel = df_green \
    .select(common_colums) \
    .withColumn('service_type', F.lit('green'))

df_yellow_sel = df_yellow \
    .select(common_colums) \
    .withColumn('service_type', F.lit('yellow'))

In [23]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [25]:
df_trips_data.groupBy("service_type").count().show()

[Stage 12:===============================================>        (17 + 3) / 20]

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [26]:
df_trips_data.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge',
 'service_type']

In [27]:
df_trips_data.registerTempTable('trips_data')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [29]:
spark.sql("""
SELECT
    service_type,
    count(*) as count
FROM
    trips_data
GROUP BY 
    service_type
""").show()

[Stage 18:============================================>           (16 + 4) / 20]

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [31]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1,2,3
""")

In [32]:
df_result.show()

[Stage 21:=====================================================>  (19 + 1) / 20]

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|          20

In [33]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')

In [34]:
spark.stop()